# 🤖 Wie arbeitet ein agentisches System?

## Lernziel

In diesem Notebook erleben wir, wie ein **agentisches System** funktioniert – und zwar an einem konkreten Beispiel: Wir geben einem Sprachmodell den Auftrag, einen Ordner auf dem Computer anzulegen.

---

## Was ist der Unterschied zwischen einem normalen Chatbot und einem Agenten?

| Normaler Chatbot | Agent |
|---|---|
| Gibt **Text als Antwort** | Führt **echte Aktionen** aus |
| "Erstelle einen Ordner namens Test" → erklärt wie man das macht | "Erstelle einen Ordner namens Test" → **legt den Ordner tatsächlich an** |
| Passiv – wartet, dass der Mensch handelt | Aktiv – handelt selbst |

---

## Der Ablauf eines agentischen Systems

```
┌─────────────────────────────────────────────────────────┐
│                                                         │
│   Mensch gibt Prompt ein                                │
│          ↓                                              │
│   Sprachmodell entscheidet:                             │
│   "Welches Werkzeug brauche ich? Mit welchen Werten?"   │
│          ↓                                              │
│   Python führt das Werkzeug aus                         │
│          ↓                                              │
│   Ergebnis: Ordner wurde erstellt ✅                    │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

Das Modell **denkt**, Python **handelt**.

---

## Schritt 1: Setup – Bibliotheken installieren und laden

Wir brauchen zwei Dinge:
- **`anthropic`**: Die offizielle Python-Bibliothek, um mit dem Claude-Sprachmodell zu kommunizieren
- **`os`**: Ein Standard-Python-Modul, das uns Zugriff auf das Dateisystem gibt
- **`python-dotenv`**: Damit wir den API-Key sicher aus einer `.env`-Datei laden können

In [ ]:
# Bibliotheken installieren (nur beim ersten Mal nötig)
%pip install anthropic python-dotenv --quiet

In [ ]:
import os                          # Zugriff auf das Dateisystem
import anthropic                   # Verbindung zum Claude-Sprachmodell
from dotenv import load_dotenv     # API-Key sicher laden

# API-Key aus der .env-Datei laden
# (Die .env-Datei liegt im gleichen Ordner wie dieses Notebook
#  und enthält eine Zeile: ANTHROPIC_API_KEY=sk-ant-...)
load_dotenv()

# Verbindung zum Sprachmodell herstellen
client = anthropic.Anthropic()

print("✅ Setup abgeschlossen – Verbindung zum Sprachmodell bereit.")

---

## Schritt 2: Das Werkzeug – eine Python-Funktion zum Ordner erstellen

Ein Agent braucht **Werkzeuge** – also konkrete Aktionen, die er ausführen kann.

Unser einziges Werkzeug heute: Eine Python-Funktion, die einen Ordner anlegt.

Wir testen sie zuerst **ohne das Sprachmodell** – damit klar wird: Das ist normales Python, das Modell kommt erst später.

In [ ]:
def ordner_erstellen(ordner_name):
    """
    Erstellt einen Ordner im aktuellen Verzeichnis.
    
    Parameter:
        ordner_name (str): Der Name des Ordners, der erstellt werden soll.
    
    Rückgabe:
        str: Eine Meldung, ob der Ordner erstellt wurde oder schon existiert.
    """
    
    # Sicherheitscheck: Keine Pfade wie "../geheim" erlauben
    if "/" in ordner_name or "\\" in ordner_name or ".." in ordner_name:
        return f"❌ Ungültiger Ordnername: '{ordner_name}'. Keine Pfade erlaubt."
    
    if os.path.exists(ordner_name):
        return f"ℹ️ Der Ordner '{ordner_name}' existiert bereits."
    
    os.makedirs(ordner_name)
    return f"✅ Ordner '{ordner_name}' wurde erfolgreich erstellt."


# --- Direkter Test ohne Sprachmodell ---
ergebnis = ordner_erstellen("test_ordner")
print(ergebnis)

---

## Schritt 3: Wie erfährt das Sprachmodell von unserem Werkzeug?

Das Modell kann unsere Python-Funktion nicht einfach "lesen". Wir müssen ihm eine **Beschreibung** geben – so wie eine Bedienungsanleitung.

Diese Beschreibung heißt **Tool-Definition** und ist ein Python-Dictionary mit genau festgelegtem Aufbau:

- **`name`**: Wie heißt das Werkzeug? (muss mit der Python-Funktion übereinstimmen)
- **`description`**: Was macht das Werkzeug? (Das liest das Modell, um zu entscheiden ob es das Werkzeug braucht)
- **`input_schema`**: Welche Informationen braucht das Werkzeug? (z.B. einen Ordnernamen)

In [ ]:
# Die "Bedienungsanleitung" für das Sprachmodell
werkzeuge = [
    {
        "name": "ordner_erstellen",           # Name des Werkzeugs
        "description":                         # Was macht es? (Das entscheidet das Modell)
            "Erstellt einen neuen Ordner im Dateisystem des Computers. "
            "Verwende dieses Werkzeug, wenn der Nutzer einen Ordner, "
            "ein Verzeichnis oder einen Speicherort anlegen möchte.",
        "input_schema": {                      # Welche Eingaben braucht es?
            "type": "object",
            "properties": {
                "ordner_name": {
                    "type": "string",           # Es muss ein Text sein
                    "description":             # Erklärung für das Modell
                        "Der Name des Ordners, der erstellt werden soll. "
                        "Nur einfache Namen ohne Schrägstriche."
                }
            },
            "required": ["ordner_name"]        # ordner_name ist Pflichtangabe
        }
    }
]

print("✅ Werkzeug-Beschreibung definiert.")
print(f"   → Das Modell kennt jetzt das Werkzeug: '{werkzeuge[0]['name']}'")

---

## Schritt 4: Der Agent in Aktion – Demo mit festem Prompt

Jetzt kommt das Herzstück. Wir schauen uns **jeden einzelnen Schritt** des Agenten an:

1. Der Prompt geht ans Modell
2. Das Modell entscheidet: "Ich brauche das Werkzeug `ordner_erstellen`"
3. Das Modell gibt zurück: welches Werkzeug mit welchem Ordnernamen
4. Python führt die Funktion aus

Wir machen alle Zwischenschritte **sichtbar**.

In [ ]:
# ─── SCHRITT 1: Prompt festlegen ───────────────────────────────────────────────

prompt = "Bitte erstelle einen Ordner namens 'mein_erster_agent_ordner'."

print("═" * 60)
print("SCHRITT 1: Prompt wird ans Sprachmodell gesendet")
print("═" * 60)
print(f"Prompt: {prompt}")
print()

In [ ]:
# ─── SCHRITT 2: Das Sprachmodell entscheidet ───────────────────────────────────

print("═" * 60)
print("SCHRITT 2: Sprachmodell denkt nach ...")
print("═" * 60)

antwort = client.messages.create(
    model="claude-sonnet-4-6",          # Das Sprachmodell, das wir verwenden
    max_tokens=256,                      # Maximale Länge der Antwort
    tools=werkzeuge,                     # Die Werkzeuge, die das Modell nutzen darf
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(f"stop_reason: '{antwort.stop_reason}'")
print()
print("Rohe Antwort des Modells (alle Inhaltsblöcke):")
for block in antwort.content:
    print(f"  → Typ: {block.type}")
    if block.type == "text":
        print(f"     Text: {block.text}")
    elif block.type == "tool_use":
        print(f"     Werkzeug: {block.name}")
        print(f"     Parameter: {block.input}")

### 💡 Was bedeutet `stop_reason: 'tool_use'`?

Das Modell hat die Antwort **nicht als normalen Text** beendet, sondern mit `tool_use`.

Das bedeutet: *"Ich habe entschieden, dass ich ein Werkzeug brauche. Bitte führe es aus und sag mir dann das Ergebnis."*

Das Modell gibt uns also **strukturiert** zurück:
- **Welches Werkzeug** es verwenden möchte
- **Mit welchen Werten** (hier: der Ordnername)

In [ ]:
# ─── SCHRITT 3: Python interpretiert die Entscheidung und handelt ──────────────

print("═" * 60)
print("SCHRITT 3: Python führt das Werkzeug aus")
print("═" * 60)

if antwort.stop_reason == "tool_use":
    
    # Den tool_use-Block aus der Antwort heraussuchen
    for block in antwort.content:
        if block.type == "tool_use":
            
            werkzeug_name = block.name           # z.B. "ordner_erstellen"
            parameter     = block.input          # z.B. {"ordner_name": "mein_ordner"}
            
            print(f"Das Modell möchte verwenden: '{werkzeug_name}'")
            print(f"Mit dem Parameter:           {parameter}")
            print()
            
            # Das richtige Werkzeug aufrufen
            if werkzeug_name == "ordner_erstellen":
                ergebnis = ordner_erstellen(parameter["ordner_name"])
                print(ergebnis)

else:
    # Das Modell hat kein Werkzeug gewählt – es hat nur geantwortet
    for block in antwort.content:
        if block.type == "text":
            print(f"Das Modell hat nur Text geantwortet: {block.text}")

---

## Schritt 5: Jetzt bist du dran – freie Eingabe

Jetzt kannst du selbst einen Prompt eingeben. Das Modell entscheidet, was es tun soll.

**Probiere verschiedene Prompts aus:**
- `"Leg bitte einen Ordner für meine Fotos an."`
- `"Erstelle ein Verzeichnis namens 'Hausaufgaben'."`
- `"Wie geht es dir?"` ← Was passiert hier wohl?

In [ ]:
# ─── Deine eigene Eingabe ──────────────────────────────────────────────────────

mein_prompt = input("Dein Prompt: ")

print()
print("═" * 60)
print(f"Prompt gesendet: '{mein_prompt}'")
print("═" * 60)

antwort2 = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=256,
    tools=werkzeuge,
    messages=[
        {"role": "user", "content": mein_prompt}
    ]
)

print(f"stop_reason: '{antwort2.stop_reason}'")
print()

if antwort2.stop_reason == "tool_use":
    for block in antwort2.content:
        if block.type == "tool_use":
            print(f"Das Modell möchte verwenden: '{block.name}'")
            print(f"Mit dem Parameter:           {block.input}")
            print()
            if block.name == "ordner_erstellen":
                ergebnis = ordner_erstellen(block.input["ordner_name"])
                print(ergebnis)
else:
    print("Das Modell hat kein Werkzeug gewählt. Antwort:")
    for block in antwort2.content:
        if block.type == "text":
            print(f"{block.text}")

---

## Schritt 6: Reflexion – Nachdenken über agentische Systeme

Wir haben gerade ein einfaches agentisches System gebaut. Zeit, darüber nachzudenken.

---

### 🔍 Fragen zum Nachdenken

**1. Was passiert, wenn der Prompt keinen Ordner erwähnt?**  
Probiere: *"Was ist die Hauptstadt von Bayern?"*  
→ Das Modell wählt kein Werkzeug (`stop_reason: end_turn`). Es antwortet nur mit Text.

**2. Was passiert, wenn der Ordner schon existiert?**  
→ Unsere Funktion gibt eine Meldung aus, statt abzubrechen. Das ist gute Fehlerbehandlung.

**3. Welche anderen Werkzeuge wären denkbar?**  
→ `datei_erstellen`, `datei_loeschen`, `internet_suche`, `email_senden` ...  
→ Je mehr Werkzeuge, desto mächtiger – und gefährlicher – der Agent.

**4. Welche Risiken hat ein System, das selbst handelt?**  
→ Was wäre, wenn das Werkzeug `dateien_loeschen` heißen würde?  
→ Wer ist verantwortlich, wenn der Agent einen Fehler macht?  
→ Warum ist unser Sicherheitscheck auf `..` in Ordnernamen wichtig?

---

### 📌 Zusammenfassung: Die drei Phasen eines Agenten

| Phase | Wer handelt | Was passiert |
|---|---|---|
| **Verstehen** | Sprachmodell | Liest den Prompt, entscheidet ob und welches Werkzeug nötig ist |
| **Entscheiden** | Sprachmodell | Gibt strukturiert zurück: Werkzeugname + Parameter |
| **Handeln** | Python | Führt die Funktion aus, verändert die reale Welt |

Das Modell denkt. Python handelt. **Zusammen sind sie ein Agent.**